In [11]:
from collections import Counter, defaultdict
import random

X_train = [
    ["Y", "N", "N", "N"],
    ["N", "Y", "N", "N"],
    ["Y", "Y", "N", "N"],
    ["Y", "N", "Y", "Y"],
    ["N", "Y", "Y", "N"]
]

y_train = ["-", "-", "+", "-", "+"]

X_test = [
    ["Y", "Y", "Y", "N"],
    ["N", "N", "N", "N"],
    ["Y", "N", "N", "Y"]
]

y_test = ["+", "-", "-"]


def class_error(y_true, y_pred):
    return sum(a != p for a, p in zip(y_true, y_pred)) / len(y_true)

In [12]:
class MajorityVoteClassifier:

    def __init__(self):
        self.majority_label = None

    def fit(self, X, y):
        self.majority_label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.majority_label] * len(X)


majority = MajorityVoteClassifier()
majority.fit(X_train, y_train)

train_pred = majority.predict(X_train)
test_pred = majority.predict(X_test)

print("majority vote")
print("training error:", class_error(y_train, train_pred))
print("test error:", class_error(y_test, test_pred))

majority vote
training error: 0.4
test error: 0.3333333333333333


In [13]:
class MemorizerClassifier:

    def __init__(self):
        self.memory = {}
        self.labels = []

    def fit(self, X, y):
        self.memory = dict(zip(map(tuple, X), y))
        self.labels = list(set(y))
        return self

    def predict(self, X):
        return [
            self.memory.get(tuple(x), random.choice(self.labels))
            for x in X
        ]


memorizer = MemorizerClassifier()
memorizer.fit(X_train, y_train)

train_pred = memorizer.predict(X_train)

print("\nmemorizer")
print("training error:", class_error(y_train, train_pred))

X_unseen = [
    ["N", "N", "Y", "Y"],
    ["Y", "Y", "Y", "Y"]
]

print("unseen predictions:", memorizer.predict(X_unseen))


memorizer
training error: 0.0
unseen predictions: ['-', '+']


In [14]:
class DecisionStumpClassifier:

    def __init__(self, feature_index=0):
        self.feature_index = feature_index
        self.rules = {}
        self.default_label = None

    def fit(self, X, y):
        groups = defaultdict(list)

        for x, label in zip(X, y):
            groups[x[self.feature_index]].append(label)

        self.rules = {
            value: Counter(labels).most_common(1)[0][0]
            for value, labels in groups.items()
        }

        self.default_label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [
            self.rules.get(x[self.feature_index], self.default_label)
            for x in X
        ]


stump2 = DecisionStumpClassifier(feature_index=2)
stump2.fit(X_train, y_train)

pred2 = stump2.predict(X_train)

print("\ndecision stump - feature 2")
print("training error:", class_error(y_train, pred2))


stump3 = DecisionStumpClassifier(feature_index=3)
stump3.fit(X_train, y_train)

pred3 = stump3.predict(X_train)

print("decision stump - feature 3")
print("training error:", class_error(y_train, pred3))


decision stump - feature 2
training error: 0.4
decision stump - feature 3
training error: 0.4


In [15]:

majority = MajorityVoteClassifier()
majority.fit(X_train, y_train)
memorizer = MemorizerClassifier()
memorizer.fit(X_train, y_train)
stump = DecisionStumpClassifier(feature_index=2)
stump.fit(X_train, y_train)
models = {
"Majority Vote": majority,
"Memorizer": memorizer,
"Decision Stump": stump
}
for name, model in models.items():
  train_pred = model.predict(X_train)

  test_pred = model.predict(X_test)
  train_error = class_error(y_train, train_pred)
  test_error = class_error(y_test, test_pred)
  print(name)
  print("Training error:", train_error)
  print("Test error:", test_error)
  print()

Majority Vote
Training error: 0.4
Test error: 0.3333333333333333

Memorizer
Training error: 0.0
Test error: 0.6666666666666666

Decision Stump
Training error: 0.4
Test error: 0.3333333333333333

